# HAAM-Enhanced Attention U-Net with Self-Organizing Neural Networks

This notebook implements the HAAM self-ONN Attention U-Net used for automated pancreas
segmentation from CT images.

## Overview

The model combines:

- Attention U-Net encoder-decoder architecture
- Hybrid Adaptive Attention Module (HAAM)
- Self-Organizing Neural Network (Self-ONN) convolution layers
- Dice + Binary Cross-Entropy loss

The notebook covers:

1. Environment setup
2. Reproducibility
3. Model architecture
4. Dataset preparation
5. Training (5-fold CV)
6. Metric evaluation

> **Note:** This notebook contains the implementation used for the HAAM-ONN
> experiment. Dataset preprocessing and patient-level splitting are documented
> separately in the project README.

# Imports

In [ ]:
# Install fastonn
!pip install git+https://github.com/junaidmalik09/fastonn.git

In [ ]:
import os
import random
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.model_selection import GroupKFold

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    jaccard_score,
    precision_score,
    recall_score,
)

from fastonn import SelfONN2d

# Reproducibility

In [ ]:
def set_seed(seed: int = 42) -> None:
    """Set random seeds for reproducible experiments."""
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

        # Reproducibility settings
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

# Image / training configuration
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8
EPOCHS = 100
LEARNING_RATE = 0.5e-5
N_FOLDS = 5


# Output directory
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset paths
IMAGE_DIR = Path("/path/to/train/images")
MASK_DIR = Path("/path/to/train/masks")

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

# Model architecture

In [ ]:
class ChannelAttentionBlockONN(nn.Module):
    """Channel attention using Self-ONN feature extraction."""

    def __init__(self, in_channels: int, reduction_ratio: int = 16):
        super().__init__()

        # Multi-scale feature extraction
        self.conv1 = SelfONN2d(
            in_channels,
            in_channels,
            kernel_size=3,
            padding=3,
            dilation=3,
            q=2,
        )

        self.conv2 = SelfONN2d(
            in_channels,
            in_channels,
            kernel_size=5,
            padding=2,
            q=2,
        )

        self.bn1 = nn.BatchNorm2d(in_channels)
        self.bn2 = nn.BatchNorm2d(in_channels)

        # Global channel descriptor
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)

        # Channel attention MLP
        attention_channels = 2 * in_channels
        hidden_channels = attention_channels // reduction_ratio

        self.fc1 = nn.Linear(
            attention_channels,
            hidden_channels,
            bias=False,
        )

        self.fc2 = nn.Linear(
            hidden_channels,
            attention_channels,
            bias=False,
        )

        # Feature fusion
        self.fusion = SelfONN2d(
            attention_channels,
            in_channels,
            kernel_size=1,
            q=2,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # Multi-scale feature extraction
        feature_1 = F.relu(self.bn1(self.conv1(x)))
        feature_2 = F.relu(self.bn2(self.conv2(x)))

        # Concatenate features
        combined = torch.cat((feature_1, feature_2), dim=1)

        # Global channel descriptor
        batch_size, channels, _, _ = combined.shape

        pooled = self.global_avg_pool(combined).view(
            batch_size,
            channels,
        )

        pooled = F.layer_norm(
            pooled,
            pooled.shape[1:],
        )

        # Generate attention weights
        attention = self.fc1(pooled)
        attention = F.relu(attention)
        attention = torch.sigmoid(self.fc2(attention))
        attention = attention.view(
            batch_size,
            channels,
            1,
            1,
        )

        # Dual attention maps
        attention_1 = attention[:, :feature_1.size(1)]
        attention_2 = 1 - attention[:, feature_1.size(1):]

        # Apply channel attention
        feature_1 = feature_1 * attention_1
        feature_2 = feature_2 * attention_2

        # Fuse features
        output = torch.cat((feature_1, feature_2), dim=1)
        output = F.relu(self.fusion(output))

        return output

In [ ]:
class SpatialAttentionBlockONN(nn.Module):
    """Spatial attention over the feature map."""

    def __init__(self, in_channels: int):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            1,
            kernel_size=7,
            padding=3,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attention = torch.sigmoid(self.conv(x))
        return x * attention

In [ ]:
class HAAMONN(nn.Module):
    """Hybrid Adaptive Attention Module."""

    def __init__(self, in_channels: int):
        super().__init__()

        self.channel_attention = ChannelAttentionBlockONN(in_channels)
        self.spatial_attention = SpatialAttentionBlockONN(in_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

In [ ]:
class AttUNetHAAMONN(nn.Module):
    """
    Attention U-Net enhanced with HAAM and Self-ONN layers.
    """

    def __init__(
        self,
        img_channels: int = 3,
        output_channels: int = 1,
        base_filters: int = 64,
    ):
        super().__init__()

        filters = [
            base_filters,
            base_filters * 2,
            base_filters * 4,
            base_filters * 8,
            base_filters * 16,
        ]

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # ---------------- Encoder ----------------

        self.encoder1 = self._encoder_block(
            img_channels,
            filters[0],
        )

        self.encoder2 = self._encoder_block(
            filters[0],
            filters[1],
        )

        self.encoder3 = self._encoder_block(
            filters[1],
            filters[2],
        )

        self.encoder4 = self._encoder_block(
            filters[2],
            filters[3],
        )

        self.encoder5 = self._encoder_block(
            filters[3],
            filters[4],
        )

        # ---------------- Decoder ----------------

        self.up5 = nn.ConvTranspose2d(
            filters[4],
            filters[3],
            kernel_size=2,
            stride=2,
        )

        self.decoder5 = self._decoder_block(
            filters[4],
            filters[3],
        )

        self.up4 = nn.ConvTranspose2d(
            filters[3],
            filters[2],
            kernel_size=2,
            stride=2,
        )

        self.decoder4 = self._decoder_block(
            filters[3],
            filters[2],
        )

        self.up3 = nn.ConvTranspose2d(
            filters[2],
            filters[1],
            kernel_size=2,
            stride=2,
        )

        self.decoder3 = self._decoder_block(
            filters[2],
            filters[1],
        )

        self.up2 = nn.ConvTranspose2d(
            filters[1],
            filters[0],
            kernel_size=2,
            stride=2,
        )

        self.decoder2 = self._decoder_block(
            filters[1],
            filters[0],
        )

        self.output = nn.Conv2d(
            filters[0],
            output_channels,
            kernel_size=1,
        )

    @staticmethod
    def _encoder_block(
        in_channels: int,
        out_channels: int,
    ) -> nn.Sequential:

        return nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(out_channels),
            nn.Sigmoid(),
            HAAMONN(out_channels),
        )

    @staticmethod
    def _decoder_block(
        in_channels: int,
        out_channels: int,
    ) -> nn.Sequential:

        return nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(out_channels),
            nn.Sigmoid(),
            HAAMONN(out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # Encoder
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))
        e5 = self.encoder5(self.pool(e4))

        # Decoder
        d5 = self.up5(e5)
        d5 = torch.cat((e4, d5), dim=1)
        d5 = self.decoder5(d5)

        d4 = self.up4(d5)
        d4 = torch.cat((e3, d4), dim=1)
        d4 = self.decoder4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat((e2, d3), dim=1)
        d3 = self.decoder3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat((e1, d2), dim=1)
        d2 = self.decoder2(d2)

        return self.output(d2)

# Dataset preprocessing

In [ ]:
def get_augmentation_pipeline(probability=0.5, image_size=(256, 256)):
    """
    Define the augmentation pipeline using Albumentations.

    Args:
        probability (float): Probability of applying each augmentation.
        image_size (tuple): Target size for resizing images.

    Returns:
        A.Compose: Albumentations augmentation pipeline.
    """
    return A.Compose([
        A.HorizontalFlip(p=0.7),  # Increased probability for flipping
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=30, p=probability),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=probability),
        A.CoarseDropout(max_holes=8, max_height=16, max_width=16, fill_value=0, p=0.5),  # Replaced Cutout with CoarseDropout
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # Normalize
        A.Resize(height=image_size[0], width=image_size[1]),  # Resize to target size
        ToTensorV2(),  # Convert to PyTorch tensor
    ])


# Define Albumentations augmentation pipeline for training
def get_training_augmentations(target_size=(256, 256)):
    """
    Define the augmentation pipeline for training.

    Args:
        target_size (tuple): Target size for resizing images.

    Returns:
        A.Compose: Albumentations augmentation pipeline for training.
    """
    return get_augmentation_pipeline(probability=0.5, image_size=target_size)


# Define Albumentations augmentation pipeline for validation/testing - ONLY Resize and Normalize
def get_validation_augmentations(target_size=(256, 256)):
    """
    Define the augmentation pipeline for validation/testing.

    Args:
        target_size (tuple): Target size for resizing images.

    Returns:
        A.Compose: Albumentations augmentation pipeline for validation/testing.
    """
    return A.Compose([
        A.Resize(height=target_size[0], width=target_size[1]),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


# Function to apply augmentations
def apply_augmentations(image, mask, augmentations):
    """
    Apply Albumentations augmentations to an image and mask.

    Args:
        image (np.ndarray): Input image (H, W, C).
        mask (np.ndarray): Corresponding mask (H, W).
        augmentations (A.Compose): Albumentations augmentation pipeline.

    Returns:
        torch.Tensor: Augmented image.
        torch.Tensor: Augmented mask.
    """
    augmented = augmentations(image=image, mask=mask)
    augmented_image = augmented["image"]
    augmented_mask = augmented["mask"]
    return augmented_image, augmented_mask

# Dataset preparation

In [ ]:
class BiomedicalDataset(Dataset):
    """Dataset for paired medical images and segmentation masks."""

    def __init__(
        self,
        image_dir: Path,
        mask_dir: Path,
        augmentations: A.Compose = None,
    ):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.augmentations = augmentations

        self.image_paths = sorted(self.image_dir.iterdir())
        self.mask_paths = sorted(self.mask_dir.iterdir())

        if len(self.image_paths) != len(self.mask_paths):
            raise ValueError(
                "Number of images and masks must match."
            )

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:

        image_path = self.image_paths[index]
        mask_path = self.mask_paths[index]

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )

        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE,
        )

        if image is None:
            raise FileNotFoundError(
                f"Could not read image: {image_path}"
            )

        if mask is None:
            raise FileNotFoundError(
                f"Could not read mask: {mask_path}"
            )

        # Convert mask to binary
        mask = (mask > 127).astype(np.float32)

        if self.augmentations is not None:
            transformed = self.augmentations(
                image=image,
                mask=mask,
            )

            image = transformed["image"]
            mask = transformed["mask"]

        # ToTensorV2 may return H x W or C x H x W
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        elif mask.ndim == 3 and mask.shape[0] != 1:
            mask = mask.permute(2, 0, 1)

        return {
            "image": image.float(),
            "mask": mask.float(),
        }

In [ ]:
dataset = BiomedicalDataset(
    image_dir=IMAGE_DIR,
    mask_dir=MASK_DIR,
    augmentations=get_training_augmentations(target_size=(256, 256))
)

print(f"Total samples: {len(dataset):,}")

# Metrices definition

In [ ]:
EPS = 1e-7

def dice_score(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    intersection = torch.sum(y_true * y_pred)

    denominator = (
        torch.sum(y_true)
        + torch.sum(y_pred)
    )

    return (
        (2 * intersection + EPS)
        / (denominator + EPS)
    ).item()


def area_error_ratio(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
) -> float:

    true_area = torch.sum(y_true).item()
    pred_area = torch.sum(y_pred).item()

    return abs(true_area - pred_area) / (true_area + EPS)


def mean_absolute_error(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
) -> float:

    return torch.mean(
        torch.abs(y_true - y_pred)
    ).item()


def specificity(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
) -> float:

    tn = torch.sum(
        (y_true == 0) & (y_pred == 0)
    ).item()

    fp = torch.sum(
        (y_true == 0) & (y_pred == 1)
    ).item()

    return tn / (tn + fp + EPS)


def tversky_index(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    alpha: float = 0.7,
    beta: float = 0.3,
) -> float:

    tp = torch.sum(y_true * y_pred).item()
    fp = torch.sum((1 - y_true) * y_pred).item()
    fn = torch.sum(y_true * (1 - y_pred)).item()

    return (
        (tp + EPS)
        / (
            tp
            + alpha * fp
            + beta * fn
            + EPS
        )
    )


def calculate_metrics(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
) -> dict[str, float]:

    y_true_flat = (
        y_true.detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    y_pred_flat = (
        y_pred.detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    unique_true = np.unique(y_true_flat)

    metrics = {
        "accuracy": accuracy_score(
            y_true_flat,
            y_pred_flat,
        ),
        "precision": precision_score(
            y_true_flat,
            y_pred_flat,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true_flat,
            y_pred_flat,
            zero_division=0,
        ),
        "f1_score": f1_score(
            y_true_flat,
            y_pred_flat,
            zero_division=0,
        ),
        "specificity": specificity(
            y_true,
            y_pred,
        ),
        "dice_similarity_coefficient": dice_score(
            y_true,
            y_pred,
        ),
        "area_error_ratio": area_error_ratio(
            y_true,
            y_pred,
        ),
        "mean_absolute_error": mean_absolute_error(
            y_true,
            y_pred,
        ),
        "mean_intersection_over_union": jaccard_score(
            y_true_flat,
            y_pred_flat,
            zero_division=0,
        ),
        "tversky_index": tversky_index(
            y_true,
            y_pred,
        ),
    }

    return metrics

# Loss definition

In [ ]:
class DiceLoss(nn.Module):
    """Dice loss for binary segmentation."""

    def forward(
        self,
        logits: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:

        probabilities = torch.sigmoid(logits)

        probabilities = probabilities.reshape(-1)
        target = target.reshape(-1)

        intersection = torch.sum(
            probabilities * target
        )

        return 1 - (
            (2 * intersection + EPS)
            / (
                torch.sum(probabilities)
                + torch.sum(target)
                + EPS
            )
        )


class CombinedLoss(nn.Module):
    """Binary cross-entropy + Dice loss."""

    def __init__(self):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(
        self,
        logits: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:

        return (
            self.bce(logits, target)
            + self.dice(logits, target)
        )

# Model, Optimizer and Scheduler

In [ ]:
model = AttUNetHAAMONN(
    img_channels=3,
    output_channels=1,
    base_filters=64,
).to(DEVICE)

criterion = CombinedLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=10,
)

use_amp = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp,
)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Training (5-fold cv)

In [ ]:
def train_one_fold(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs,
    fold,
    output_dir,
):
    """Train and validate one cross-validation fold."""

    fold_dir = output_dir / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    best_val_dice = 0.0
    history = []

    best_model_path = fold_dir / "best_model.pth"
    history_path = fold_dir / "training_history.csv"

    start_time = time.time()

    for epoch in range(epochs):

        # ============================================================
        # Training
        # ============================================================

        model.train()

        running_loss = 0.0
        train_metrics = []

        progress = tqdm(
            train_loader,
            desc=f"Fold {fold} | Epoch {epoch + 1}/{epochs} | Train",
            leave=False,
        )

        for batch in progress:

            images = batch["image"].to(
                DEVICE,
                non_blocking=True,
            )

            masks = batch["mask"].to(
                DEVICE,
                non_blocking=True,
            )

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=use_amp,
            ):
                outputs = model(images)
                loss = criterion(outputs, masks)

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()

            predictions = (
                torch.sigmoid(outputs) > 0.5
            ).float()

            batch_metrics = calculate_metrics(
                masks,
                predictions,
            )

            train_metrics.append(batch_metrics)
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        avg_train_metrics = (
            pd.DataFrame(train_metrics)
            .mean()
            .to_dict()
        )

        # ============================================================
        # Validation
        # ============================================================

        model.eval()

        running_val_loss = 0.0
        val_metrics = []

        with torch.no_grad():

            progress = tqdm(
                val_loader,
                desc=f"Fold {fold} | Epoch {epoch + 1}/{epochs} | Val",
                leave=False,
            )

            for batch in progress:

                images = batch["image"].to(
                    DEVICE,
                    non_blocking=True,
                )

                masks = batch["mask"].to(
                    DEVICE,
                    non_blocking=True,
                )

                with torch.amp.autocast(
                    device_type=DEVICE.type,
                    enabled=use_amp,
                ):
                    outputs = model(images)
                    loss = criterion(outputs, masks)

                predictions = (
                    torch.sigmoid(outputs) > 0.5
                ).float()

                batch_metrics = calculate_metrics(
                    masks,
                    predictions,
                )

                val_metrics.append(batch_metrics)
                running_val_loss += loss.item()

        val_loss = running_val_loss / len(val_loader)

        avg_val_metrics = (
            pd.DataFrame(val_metrics)
            .mean()
            .to_dict()
        )

        # ============================================================
        # Save history
        # ============================================================

        epoch_results = {
            "fold": fold,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "learning_rate": optimizer.param_groups[0]["lr"],
            **{
                f"train_{key}": value
                for key, value in avg_train_metrics.items()
            },
            **{
                f"val_{key}": value
                for key, value in avg_val_metrics.items()
            },
        }

        history.append(epoch_results)

        print(
            f"Fold {fold} | "
            f"Epoch {epoch + 1:03d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Dice: "
            f"{avg_train_metrics['dice_similarity_coefficient']:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Dice: "
            f"{avg_val_metrics['dice_similarity_coefficient']:.4f}"
        )

        # ============================================================
        # Save best model
        # ============================================================

        val_dice = avg_val_metrics[
            "dice_similarity_coefficient"
        ]

        if val_dice > best_val_dice:

            best_val_dice = val_dice

            torch.save(
                model.state_dict(),
                best_model_path,
            )

            print(
                f"  ✓ Best model saved "
                f"(Val Dice: {best_val_dice:.4f})"
            )

        scheduler.step(val_loss)

        pd.DataFrame(history).to_csv(
            history_path,
            index=False,
        )

    elapsed_time = time.time() - start_time

    print(
        f"Fold {fold} completed in "
        f"{elapsed_time / 60:.2f} minutes"
    )

    return pd.DataFrame(history)

In [ ]:
def run_5_fold_cross_validation(
    dataset,
    output_dir,
    n_folds=5,
):
    """
    Perform patient-level 5-fold cross-validation.

    Parameters
    ----------
    dataset:
        Complete dataset.

    output_dir:
        Directory for fold outputs.

    n_folds:
        Number of CV folds.
    """

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    gkf = GroupKFold(
        n_splits=n_folds
    )

    all_fold_results = []
    fold_histories = []

    for fold, (train_idx, val_idx) in enumerate(
        gkf.split(
            X=np.zeros(len(dataset)),
            y=None,
        ),
        start=1,
    ):

        print("\n" + "=" * 70)
        print(f"STARTING FOLD {fold}/{n_folds}")
        print("=" * 70)

        print(
            f"Training samples:   {len(train_idx):,}"
        )

        print(
            f"Validation samples: {len(val_idx):,}"
        )

        # ------------------------------------------------------------
        # Create fold datasets
        # ------------------------------------------------------------

        train_subset = Subset(
            dataset,
            train_idx,
        )

        val_subset = Subset(
            dataset,
            val_idx,
        )

        train_loader = DataLoader(
            train_subset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=2,
            pin_memory=torch.cuda.is_available(),
        )

        val_loader = DataLoader(
            val_subset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=torch.cuda.is_available(),
        )

        # ------------------------------------------------------------
        # Create a NEW model for every fold
        # ------------------------------------------------------------

        model = AttUNetHAAMONN(
            img_channels=3,
            output_channels=1,
            base_filters=64,
        ).to(DEVICE)

        criterion = CombinedLoss()

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE,
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=10,
        )

        global scaler

        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=use_amp,
        )

        # ------------------------------------------------------------
        # Train fold
        # ------------------------------------------------------------

        history = train_one_fold(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            epochs=EPOCHS,
            fold=fold,
            output_dir=output_dir,
        )

        fold_histories.append(history)

        # ------------------------------------------------------------
        # Best fold result
        # ------------------------------------------------------------

        best_row = history.loc[
            history["val_dice_similarity_coefficient"].idxmax()
        ]

        fold_result = {
            "fold": fold,
            "best_epoch": int(
                best_row["epoch"]
            ),
            "best_val_dice": best_row[
                "val_dice_similarity_coefficient"
            ],
            "val_iou": best_row[
                "val_mean_intersection_over_union"
            ],
            "val_precision": best_row[
                "val_precision"
            ],
            "val_recall": best_row[
                "val_recall"
            ],
            "val_f1": best_row[
                "val_f1_score"
            ],
        }

        all_fold_results.append(
            fold_result
        )

    # ================================================================
    # Aggregate results
    # ================================================================

    fold_results = pd.DataFrame(
        all_fold_results
    )

    fold_results.to_csv(
        output_dir / "5_fold_results.csv",
        index=False,
    )

    numeric_columns = [
        "best_val_dice",
        "val_iou",
        "val_precision",
        "val_recall",
        "val_f1",
    ]

    summary = pd.DataFrame({
        "mean": fold_results[numeric_columns].mean(),
        "std": fold_results[numeric_columns].std(
            ddof=1
        ),
    })

    summary.to_csv(
        output_dir / "5_fold_summary.csv"
    )

    print("\n" + "=" * 70)
    print("5-FOLD CROSS-VALIDATION RESULTS")
    print("=" * 70)

    print(fold_results)

    print("\nMean ± Standard Deviation:")
    print(summary)

    return (
        fold_results,
        summary,
        fold_histories,
    )

In [ ]:
fold_results, cv_summary, fold_histories = (
    run_5_fold_cross_validation(
        dataset=dataset,
        output_dir=OUTPUT_DIR,
        n_folds=5,
    )
)